In [1]:
### Import packages
import sys, getopt, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import RepeatedKFold
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler #StandardScaler is sensitive to outlier

from qiskit.circuit.library import ZZFeatureMap
from qiskit_machine_learning.algorithms import QSVC
from qiskit_machine_learning.kernels import FidelityQuantumKernel


In [3]:
#Instead of FEATURE_MAP_REPS_LIST = [1,2,3,4,5] and REGU_PARA_LIST = [0.1,1,10,100], I used FEATURE_MAP_REPS_LIST = [1] and REGU_PARA_LIST = [0.1] to reduce the number of experiments/Iterations and save time. You can change it back to the original values if you want to run more experiments.
#Also I changed the number of repeats from N_REPEATS = 10 to N_REPEATS = 1 for the same reason as above. You can change it back to the original value if you want to run more experiments.
#By my calculations, the total number of experiments/Iterations will have 210 folds and takes roughly 21 hours by my worskstation, if you use the original values. If you use the reduced values, the total number of experiments/Iterations will have 21 folds and takes roughly 1.2 minutes.
#This can be used to test the code and make sure it works, and then you can change it back to the original values to run the full set of experiments.

In [4]:
#Rules of the experiment
root_folder = 'QSVC'
### Globals
# For reproducibility
np.random.seed(42)

# Fixed feature sizes
NUM_FEATURES = 3
NUM_QUBITS = NUM_FEATURES
NUM_TARGETS = 1

# Quantum circuit parameters
FEATURE_MAP_REPS_LIST = [1,2,3,4,5]
REGU_PARA_LIST = [0.1,1,10,100]
ENTANGLEMENT_LIST = ['linear', 'full', 'circular']

# Training hyperparameters
#LEARNING_RATE = 0.01
#BATCH_SIZE = 30
#NUM_EPOCHS = 100 # Adjust as needed

# K-fold cross-validation parameters
N_REPEATS = 10
TEST_SIZE = 1

# Data conf
CLASSIFIER_THRESHOLD = 19


In [6]:
#Dataset preparation
def prepare_dataset_k_fold(X, y, train_indices, test_indices):
    # Separate train/test split
    X_train_raw, X_test_raw = X[train_indices], X[test_indices]
    y_train, y_test = y[train_indices], y[test_indices]

    # Separate element column from the actual features
    element_test = X_test_raw[:, 0]
    element_train = X_train_raw[:, 0]

    # Drop the element column (first column)
    X_train = X_train_raw[:, 1:]
    X_test = X_test_raw[:, 1:]

    full_X = np.vstack([X_train, X_test])

    scaler = MinMaxScaler(feature_range=(-1, 1))
    scaler.fit(full_X)

    X_train_scaled = scaler.transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    return X_train_scaled, y_train, X_test_scaled, y_test, element_test, element_train


In [7]:
#It prepares one train-test split for cross-validation by separating the data, removing unnecessary columns, scaling the numerical features, and returning everything needed for training and evaluation.
#Arguments = (X, y, train_indices, test_indices)

#X.This is the input feature matrix.Imagine the dataset before processing.
#Element	Electronegativity	Bulk Modulus	Volume
#Mg-Al	          1.42	             37	         14.2
#Mg-Zn	          1.65	             41	         13.8
#Mg-Y	          1.22	             28	         18.1

#y.This contains the labels.
#For classification:
#0
#1
#0
#1
#These are the answers the model is trying to learn.

#train_indices
#Suppose Repeated K-Fold decides
# Training, Samples
#0
#2
#3qml_training-validation-data.csvqml_training-validation-data.csv
#5
#6
#8

#It doesn't copy those rows.Instead it simply stores [0,2,3,5,6,8].
#Those are the training indices.

#test_indices
#Similarly, [1,4,7] means
#Rows
#1
#4
#7
#are the testing set.

#X_train_raw, X_test_raw = X[train_indices], X[test_indices]
# y_train, y_test = y[train_indices], y[test_indices]
#Train Test split

#   element_test = X_test_raw[:, 0]
#   element_train = X_train_raw[:, 0](Splicing)
#   Only returns the first coloumn of data, which is the element name

#   X_train = X_train_raw[:, 1:]
#    X_test = X_test_raw[:, 1:]
#removes the first coloumn, which has the names of the elements(strings) from the further calculations.

#full_X = np.vstack([X_train, X_test])
#np.vstack means vertical stack.
#Training
#1 2 3
#4 5 6
#Testing
#7 8 9

#After
#np.vstack(...)
#you obtain
#1 2 3
#4 5 6
#7 8 9
#The two datasets are temporarily joined together.

# scaler = MinMaxScaler(feature_range=(-1, 1))
# This creates a MinMaxScaler object.It doesn't scale anything yet.
# Think of it like buying a ruler.The ruler exists, but you haven't measured anything.
#                   Feature	            Range 
#               Electronegativity	    1–4
#                Bulk Modulus	        20–250
#                    Volume	            10–30

#Notice that one feature has values around 200 while another is around 2.
#If we leave them like this, larger numerical ranges can dominate the calculations.
#Scaling brings all features onto a common scale.

# scaler.fit(full_X)
# This line computes the minimum and maximum values of each feature.The scaler stores these values internally.
# fit() does not modify the data. It only learns the transformation.

#X_train_scaled = scaler.transform(X_train)
#X_test_scaled = scaler.transform(X_test)
#the learned scaling is applied to the training features.The same transformation is then applied to the test features.
#Using the same scaler is essential so that training and testing data remain in the same feature space.

#The choice to fit the scaler on full_X (training + testing) is perfectly acceptable to reproduce the published work exactly
#But if you later develop your own improved methodology, one potential enhancement would be to fit the scaler using only the training data within each cross-validation fold. 
#Recognizing these small design decisions is an important step from simply reproducing code to critically evaluating and eventually improving it.

In [8]:
#Quantum kernel builder
def reconfig_quantum_kernel_qsvc(feature_dimension, C, reps, entangle):
    """
        Create a quantum kernel
        qsvc = QSVC(C=20.0, quantum_kernel=kernel)

        Args:
            feature_dimension: Dimension of the feature space.
            reps: Number of repetitions of quantum circuit.
            C: Regularization parameter.
               The strength of the regularization is inversely proportional to C.
               Must be strictly positive. The penalty is a squared l2.
            entangle: Entanglement type of the feature map.

        Returns:
            qsvc: quantum kernel
    """
    feature_map = ZZFeatureMap(feature_dimension=feature_dimension, reps=reps, entanglement=entangle, insert_barriers=True)
    kernel = FidelityQuantumKernel(feature_map=feature_map)
    qsvc = QSVC(C=C, quantum_kernel=kernel)
    return qsvc

In [10]:
#Training function (Very Important)
def train_qsvc(qsvc, X_train, y_train, X_test):
    """
        Train based on X_train/y_train (after scaling), return prediction from X_test

        Args:
            qsvc: quantum kernel
            X_train:
            y_train
            X_test

        Returns:
    """
    qsvc.fit(X_train, np.concatenate(y_train))
    return qsvc.predict(X_train), qsvc.predict(X_test)

In [12]:
#command line argument parser
def get_arguments(argvs):
    _entangle = ''
    _feature_map_reps = ''
    _regu_para = ''
    try:
        opts, args = getopt.getopt(argvs, "h:e:f:r:", ["entangle=", "feature_map_reps=", "_regu_para="])
    except getopt.GetoptError:
        print(root_folder + '.py -e <entangle> -f <feature_map_reps> -r <regu_para>')
        sys.exit(2)
    for opt, arg in opts:
        if opt == '-h':
            print(root_folder + '.py -e <entangle> -f <feature_map_reps> -r <regu_para>')
            sys.exit()
        elif opt in ("-e", "--entangle"):
            _entangle = arg
        elif opt in ("-f", "--feature_map_reps"):
            _feature_map_reps = int(arg)
        elif opt in ("-r", "--regu_para"):
            _regu_para = float(arg)
    return _entangle, _feature_map_reps, _regu_para

In [14]:
#output folder
date = "28_19_25_1"

if not os.path.exists(f'{root_folder}/result'):
    os.makedirs(f'{root_folder}/result')
if not os.path.exists(f'{root_folder}/logs'):
    os.makedirs(f'{root_folder}/logs')

In [17]:
#Load dataset
dataset_name = "qml_training-validation-data.csv"
df = pd.read_csv(dataset_name)
display(df.head())
X = df[['Element', 'el_neg', 'B/GPa', 'Volume/A^3']].values
y = df['SFE/mJm^-3'].values
print(df.shape)

,Element,el_neg,B/GPa,Volume/A^3,SFE/mJm^-3
0,Be,1.57,130.0,8.09,23.48
1,Sc,1.36,57.0,25.00,16.16
2,Ti,1.54,110.0,17.60,24.44
3,Co,1.88,180.0,11.00,37.64
4,Zn,1.65,70.0,15.20,20.98


(21, 5)


In [19]:
# Regression to classification conversion
for i in range(0, len(y)):
    if y[i] > CLASSIFIER_THRESHOLD:
        y[i] = 0
    else:
        y[i] = 1

y_scaler = MinMaxScaler(feature_range=(-1, 1))
y = y_scaler.fit_transform(y.reshape(-1, 1))


In [21]:
#Cross-validation through RepeatedKFold
rkf = RepeatedKFold(n_splits=X.shape[0] // TEST_SIZE, n_repeats=N_REPEATS)
print(rkf)

RepeatedKFold(n_repeats=10, n_splits=21, random_state=None)


In [23]:
#Results dataframe
df = pd.DataFrame(columns=['C', 'reps', 'entanglement',
                               'element test', 'actual test', 'predicted test',
                               'element train', 'actual train', 'predicted train',
                               'R2 test', 'R2 train'])


In [25]:
# Build output filename

if len(FEATURE_MAP_REPS_LIST) == 1:
    FEATURE_MAP_REPS_LIST_NAME = FEATURE_MAP_REPS_LIST[0]
else:
    FEATURE_MAP_REPS_LIST_NAME = FEATURE_MAP_REPS_LIST

if len(REGU_PARA_LIST) == 1:
    REGU_PARA_LIST_NAME = REGU_PARA_LIST[0]
else:
    REGU_PARA_LIST_NAME = REGU_PARA_LIST

if len(ENTANGLEMENT_LIST) == 1:
    ENTANGLEMENT_LIST_NAME = ENTANGLEMENT_LIST[0]
else:
    ENTANGLEMENT_LIST_NAME = ENTANGLEMENT_LIST


file_name = (
    f"{root_folder}/result/"
    f"FMR_{FEATURE_MAP_REPS_LIST_NAME}_"
    f"R_{REGU_PARA_LIST_NAME}_"
    f"E_{ENTANGLEMENT_LIST_NAME}_"
    f"{date}.csv"
)

print(file_name)

QSVC/result/FMR_[1, 2, 3, 4, 5]_R_[0.1, 1, 10, 100]_E_['linear', 'full', 'circular']_28_19_25_1.csv


In [27]:
i = 0

print("\n--- Start K-Fold Loop ---")

for train_indices, test_indices in rkf.split(X):
    X_train, y_train, X_test, y_test, element_test, element_train = prepare_dataset_k_fold(X, y, train_indices, test_indices)
    for C_value in REGU_PARA_LIST:
        for feature_map_reps in FEATURE_MAP_REPS_LIST:
            for entanglement in ENTANGLEMENT_LIST:
                print(f'C:{C_value} feature_map_reps:{feature_map_reps} entanglement:{entanglement}')
                # conf kernel
                qsvc = reconfig_quantum_kernel_qsvc(feature_dimension=NUM_FEATURES,
                                                    C=C_value,
                                                    reps=feature_map_reps,
                                                    entangle=entanglement)

                # train
                predict_train, predict_test = train_qsvc(qsvc, X_train, y_train, X_test)

                # some conversions
                all_preds = np.array(predict_test)
                all_targets = np.array(y_test)
                all_preds = y_scaler.inverse_transform(all_preds.reshape(-1, 1))
                all_targets = y_scaler.inverse_transform(all_targets.reshape(-1, 1))

                all_preds_train = np.array(predict_train)
                all_targets_train = np.array(y_train)
                all_preds_train = y_scaler.inverse_transform(all_preds_train.reshape(-1, 1))
                all_targets_train = y_scaler.inverse_transform(all_targets_train.reshape(-1, 1))

                # save data
                new_row = {'C': C_value,
                            'reps': feature_map_reps,
                            'entanglement': entanglement,
                            'element test': element_test,
                            'actual test': np.array(all_targets).flatten(),
                            'predicted test': np.array(all_preds).flatten(),
                            'element train': element_train,
                            'actual train': np.array(all_targets_train).flatten(),
                            'predicted train': np.array(all_preds_train).flatten(),
                            #'R2 test': r2_score(y_test, predict_test),
                            'R2 train': r2_score(y_train, predict_train),
                            }
                df.loc[len(df)] = new_row
                with np.printoptions(linewidth=10000):
                    df.to_csv(file_name, index=False)  # update csv every loop
                df.at[0, "info"] = [f"DATASET: {dataset_name}, "
                                    f"CLASSIFIER_THRESHOLD = {CLASSIFIER_THRESHOLD}"]
                i += 1


--- Start K-Fold Loop ---
C:0.1 feature_map_reps:1 entanglement:linear
C:0.1 feature_map_reps:1 entanglement:full
C:0.1 feature_map_reps:1 entanglement:circular
C:0.1 feature_map_reps:2 entanglement:linear
C:0.1 feature_map_reps:2 entanglement:full
C:0.1 feature_map_reps:2 entanglement:circular
C:0.1 feature_map_reps:3 entanglement:linear
C:0.1 feature_map_reps:3 entanglement:full
C:0.1 feature_map_reps:3 entanglement:circular
C:0.1 feature_map_reps:4 entanglement:linear
C:0.1 feature_map_reps:4 entanglement:full
C:0.1 feature_map_reps:4 entanglement:circular
C:0.1 feature_map_reps:5 entanglement:linear
C:0.1 feature_map_reps:5 entanglement:full
C:0.1 feature_map_reps:5 entanglement:circular
C:1 feature_map_reps:1 entanglement:linear
C:1 feature_map_reps:1 entanglement:full
C:1 feature_map_reps:1 entanglement:circular
C:1 feature_map_reps:2 entanglement:linear
C:1 feature_map_reps:2 entanglement:full
C:1 feature_map_reps:2 entanglement:circular
C:1 feature_map_reps:3 entanglement:line

In [ ]:
# The heart of the code
""" It contains:

* Cross Validation
* Hyperparameter Search
* Quantum Model Construction
* Model Training
* Prediction
* Result Collection
* Saving Results 
                    """

#What is the code trying to accomplish?
#"Among all possible QSVC configurations, which one classifies Mg alloys the best?"
#To answer this question,they don't train one model.They train thousands of models.

#Big picture
#Entire Dataset --> Leave-One-Out Split --> Try every C --> Try every Feature Map --> 
#Try every Entanglement -->Build QSVC --> Train QSVC --> Predict --> Store Results -->
#Repeat for next split

In [ ]:
#The First Loop
#for train_indices, test_indices in rkf.split(X):
#rkf: Every iteration gives Training Indices, Testing Indices.All samples get tested here.
# X_train, y_train, X_test, y_test, element_test, element_train = prepare_dataset_k_fold(X, y, train_indices, test_indices)
#It performs Split, Remove element names, Scale features and returns
''' Training Features

    Training Labels

    Testing Features

    Testing Labels

    Training Elements

    Testing Elements '''

In [ ]:
#The Second Loop, the loop over C
#for C_value in REGU_PARA_LIST:
#On the second cell, we've seen that REGU_PARA_LIST:[0.1,1,10,100]
#Therefore, the loop becomes four experiments
''' C = 0.1

        ↓

    C = 1

        ↓

    C = 10

        ↓

    C = 100 
                '''

In [ ]:
#The third loop, loop over repetitions
#for feature_map_reps in FEATURE_MAP_REPS_LIST:
#On the second cell, we've seen that FEATURE_MAP_REPS_LIST:[1,2,3,4,5]
#Therefore we have 5 experiments this time.

In [ ]:
#The Fourth Loop, loop over Entanglement
#for entanglement in ENTANGLEMENT_LIST:
#On the second cell, we've seen that ENTANGLEMENT_LIST:[['linear','full','circular']
#Therefore, we've 3 experiments this time.


#So , that's a total of 4x5x3 = 60 models per splits.
#Including the 21 samples, it will be 21x60 = 1260 models.
#With N_REPEATS being 10, it'll be 1260x10 = 12,600 QSVC trainings.

#print(f'C:{C_value} feature_map_reps:{feature_map_reps} entanglement:{entanglement}')
#Used to monitor the process.

'''qsvc = reconfig_quantum_kernel_qsvc(feature_dimension=NUM_FEATURES,
                                      C=C_value,
                                      reps=feature_map_reps,
                                      entangle=entanglement) '''
#We've looked it earlier.Internally it performs 
#                     ZZFeatureMap --> Quantum Kernel --> QSVC
#Every iteration builds a completely new quantum classifier.

#predict_train, predict_test = train_qsvc(qsvc, X_train, y_train, X_test)
#training to predict : Training for Learning and testing for generalization.

#all_preds = y_scaler.inverse_transform(all_preds.reshape(-1, 1))
#all_targets = y_scaler.inverse_transform(all_targets.reshape(-1, 1))
#Previously because of MinMaxScaler, the range was from -1,1. Now it changes back to 0,1
#Everything under #Some Conversions comes under this category.

## save data
''' new_row = {'C': C_value,
                'reps': feature_map_reps,
                'entanglement': entanglement,
                'element test': element_test,
                'actual test': np.array(all_targets).flatten(),
                'predicted test': np.array(all_preds).flatten(),
                'element train': element_train,
                'actual train': np.array(all_targets_train).flatten(),
                'predicted train': np.array(all_preds_train).flatten(),
                #'R2 test': r2_score(y_test, predict_test),
                'R2 train': r2_score(y_train, predict_train),
                            }'''
#This Dictionary stores everything: C, reps, entanglement, test material, prediction, everything is preserved.
#Note; only R2 test is not used here, which deepens the idea that the code was originally a regression.
#Because classification papers almost never report R².

#df.loc[len(df)] = new_row: Adds a new row to the end of a dataframe.
#Every experiment adds another row.

#df.to_csv(file_name, index=False)  # update csv every loop
#The result is saved and updated to a csv file everytime a run is completed.

#df.at[0, "info"]
#Adds extra data into the first row.




In [ ]:
#Entire pipeline of the experiment
'''Load Dataset
      │
      ▼
Cross Validation
      │
      ▼
Train/Test Split
      │
      ▼
Scale Features
      │
      ▼
For every C
      │
      ▼
For every Repetition
      │
      ▼
For every Entanglement
      │
      ▼
Build Quantum Kernel
      │
      ▼
Create QSVC
      │
      ▼
Train
      │
      ▼
Predict
      │
      ▼
Inverse Scaling
      │
      ▼
Store Results
      │
      ▼
Save CSV
      │
      ▼
Next Experiment '''

In [ ]:
#Final Conclusive Workflow
''' 
1. Load the alloy dataset.
2. Convert the SFE regression target into binary classes.
3. Use Leave-One-Out Cross Validation to evaluate fairly.
4. For every combination of C, reps, and entanglement, build a new QSVC.
5. Train it on the training alloys.
6. Predict both training and unseen test alloys.
7. Store every result in a DataFrame and continuously save it to a CSV file. '''
